# Fixed-rate bond valuation and analytics

Value a five-year USD fixed-rate bond, imply a Z-spread from its clean price,
and read yield, modified duration, cashflows and quote risk. Contract terms and
price anchors are explicit. Follow [setup](../../README.md#install-and-run),
then run every cell in order; no other notebook must run first.

In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## A straight bond: distinguish position value from a market price quote

We buy USD 1 million principal of a five-year, 4.5% fixed-rate bond. The context
selects the USD discount curve. A zero Z-spread first gives the curve-implied
value; a quoted clean price then asks which constant spread reconciles that
price with the discounted cashflows.

`BondPricingAnchor.clean_price(99.25)` means **99.25 per 100 principal**.
`result.price` remains the **USD position PV**. Yield-to-maturity and modified
duration use the bond's rate conventions; Z-spread is continuously compounded
and is added to the discount curve. These are different descriptions of the
same cashflows, not interchangeable input rates.

In [2]:
bond = qs.FixedRateBond(
    identifier="USD-FIXED-BOND-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    rate=0.045,
    currency=qs.Currency.USD,
    day_counter=qs.DayCounter.Actual365,
    compounding=qs.Compounding.Compounded,
    rate_frequency=qs.Frequency.Semiannual,
    payment_frequency=qs.Frequency.Semiannual,
)
trade = qs.Trade(bond, ref, pricing_anchor=qs.BondPricingAnchor.z_spread(0.0))
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
)

with context:
    curve_value = context.evaluate(trade, [qs.Request.Value])
    bond_result = context.evaluate(
        trade.with_pricing_anchor(qs.BondPricingAnchor.clean_price(99.25)),
        [
            qs.Request.Value,
            qs.Request.YieldToMaturity,
            qs.Request.ModifiedDuration,
            qs.Request.ZSpread,
            qs.Request.Cashflows,
            qs.Request.Sensitivities,
        ],
    )
    spread_value = context.evaluate(
        trade.with_pricing_anchor(qs.BondPricingAnchor.z_spread(bond_result.z_spread)),
        [qs.Request.Value],
    )

pd.Series(
    {
        "curve_only_pv_usd": curve_value.price,
        "price_anchored_pv_usd": bond_result.price,
        "spread_anchored_pv_usd": spread_value.price,
        "anchor_pv_difference_usd": spread_value.price - bond_result.price,
        "ytm": bond_result.yield_to_maturity,
        "modified_duration_years": bond_result.modified_duration,
        "z_spread_bp": bond_result.z_spread * 10_000,
    }
)

curve_only_pv_usd           1.018167e+06
price_anchored_pv_usd       9.925000e+05
spread_anchored_pv_usd      9.925000e+05
anchor_pv_difference_usd    0.000000e+00
ytm                         4.669820e-02
modified_duration_years     4.429526e+00
z_spread_bp                 5.547644e+01
dtype: float64

The spread is an independent risk factor after it has been solved. The
`<identifier>.ZSpread` row measures sensitivity to that spread; the other rows
measure curve-quote sensitivity **holding the solved spread fixed**. They do
not describe re-solving the spread to keep the observed clean price constant
under every market shock.

In [3]:
bond_result.sensitivities.assign(
    pnl_for_plus_1bp_usd=lambda frame: frame["value"] * 1e-4
)


,pillar,value,pnl_for_plus_1bp_usd
0,FixedRateDeposit_USD_SOFR_1D,-2.756600e+03,-0.275660
1,OIS_USD_SOFR_10Y,0.000000e+00,0.000000
2,OIS_USD_SOFR_1Y,-7.328608e+03,-0.732861
3,OIS_USD_SOFR_2Y,-7.142079e+03,-0.714208
4,OIS_USD_SOFR_30Y,0.000000e+00,0.000000
5,OIS_USD_SOFR_3Y,-2.408953e+04,-2.408953
6,OIS_USD_SOFR_5Y,-4.319454e+06,-431.945412
7,OIS_USD_SOFR_7Y,0.000000e+00,0.000000
8,USD-FIXED-BOND-5Y.ZSpread,-4.564245e+06,-456.424490


In [4]:
assert abs(spread_value.price - bond_result.price) < 1e-5
assert bond_result.yield_to_maturity is not None
assert bond_result.modified_duration > 0
print("The solved Z-spread reproduces the price-anchored position value.")

The solved Z-spread reproduces the price-anchored position value.


Useful straight-bond requests are `Value`, `YieldToMaturity`, `ModifiedDuration`,
`ZSpread`, `Cashflows` and `Sensitivities`. A clean price is quoted per 100;
position PV is in USD. For issuer exercise and OAS, see
[the callable stepped bond](stepped_coupon_bond.ipynb).